# The Edge — Room-Level Graph Analysis
## Degree · Closeness · Clustering · Shortest Paths

**Building:** The Edge, Carrasco, Montevideo, Uruguay  
**Architects:** Foster + Partners + Ponce de León Architects  
**Student:** Rania Chihaoui | IAAC 2025–26

---

**Graph structure (Assignment 2 methodology):**
- **Node** = one room in the floor plan
- **Edge** = shared door / direct access between two rooms
- **Graph** = one per residential unit (8 units total)

**Metrics computed per room:**

| Metric | What it measures | Architectural meaning |
|--------|------------------|-----------------------|
| **Degree Centrality** | Fraction of rooms directly connected | Identifies hub rooms (corridors, kitchens) |
| **Closeness Centrality** | How quickly a room can reach all others | Measures spatial accessibility |
| **Clustering Coefficient** | How many neighbours are also connected | Reveals clustered spatial groupings |
| **Shortest Path** | Min. number of rooms to traverse | Quantifies spatial depth / sequence |

## Section 1 — Setup & Room Program

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

renderer = "vscode"

# Zone colours matching MSD label scheme
LABEL_COLORS = {
    0: "#4e9af1",   # Day zone — blue
    1: "#f4a261",   # Night / Bedroom — orange
    2: "#e76f51",   # Night / Master — dark orange
    3: "#e9c46a",   # Night / Walk-in — yellow
    4: "#f4a261",   # Night / Service bed — orange
    5: "#2a9d8f",   # Service / Kitchen — teal
    6: "#8ecae6",   # Service / WC — light blue
    7: "#219ebc",   # Service / Bathroom — blue-teal
    8: "#adb5bd",   # Circulation — grey
}
LABEL_NAMES = {
    0: "Day (Living/Terrace)",
    1: "Bedroom",
    2: "Master Bedroom",
    3: "Walk-in Closet",
    4: "Service Bedroom",
    5: "Kitchen/Laundry",
    6: "WC/Toilette",
    7: "Bathroom",
    8: "Corridor",
}
print("Setup complete.")

In [ ]:
# ── Room program: all 8 units from the official brochure ────────────────────
# Node = room   |   Edge = shared door   |   Cross-floor edges = private stairs

UNIT_ROOMS = {
    "101": {
        "unit_type": "DUPLEX", "floors": [0, 1],
        "rooms": [
            {"name": "Estar Familiar GF",      "floor": 0, "label": 0, "area": 55},
            {"name": "Pasillo GF",             "floor": 0, "label": 8, "area": 15},
            {"name": "Dormitorio Principal 1", "floor": 0, "label": 2, "area": 30},
            {"name": "Baño Principal 1",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 1",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Principal 2", "floor": 0, "label": 2, "area": 28},
            {"name": "Baño Principal 2",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 2",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Servicio",    "floor": 0, "label": 1, "area": 18},
            {"name": "Baño Servicio",          "floor": 0, "label": 7, "area":  6},
            {"name": "Lavadero GF",            "floor": 0, "label": 5, "area": 10},
            {"name": "Jardín Privado",         "floor": 0, "label": 0, "area": 80},
            {"name": "Estar F1",               "floor": 1, "label": 0, "area": 45},
            {"name": "Comedor",                "floor": 1, "label": 0, "area": 30},
            {"name": "Cocina F1",              "floor": 1, "label": 5, "area": 20},
            {"name": "Toilette F1",            "floor": 1, "label": 6, "area":  5},
            {"name": "Lavadero F1",            "floor": 1, "label": 5, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
        ],
        "edges": [
            (0,1),(0,11),(1,2),(1,5),(1,8),(1,10),
            (2,3),(2,4),(5,6),(5,7),(8,9),
            (12,13),(12,17),(13,14),(14,15),(14,16),
            (0,12),
        ],
    },
    "102": {
        "unit_type": "1 PLANTA", "floors": [1],
        "rooms": [
            {"name": "Estar Familiar",        "floor": 1, "label": 0, "area": 60},
            {"name": "Dormitorio Principal",  "floor": 1, "label": 2, "area": 28},
            {"name": "Baño Principal",        "floor": 1, "label": 7, "area":  8},
            {"name": "Vestidor Principal",    "floor": 1, "label": 3, "area": 10},
            {"name": "Dormitorio 2",          "floor": 1, "label": 1, "area": 22},
            {"name": "Baño 2",               "floor": 1, "label": 7, "area":  6},
            {"name": "Cocina",               "floor": 1, "label": 5, "area": 20},
            {"name": "Lavadero",             "floor": 1, "label": 5, "area":  8},
            {"name": "Baño Guest",           "floor": 1, "label": 7, "area":  5},
            {"name": "Terraza",              "floor": 1, "label": 0, "area": 50},
        ],
        "edges": [(0,1),(0,4),(0,6),(0,9),(1,2),(1,3),(4,5),(6,7),(6,8)],
    },
    "103": {
        "unit_type": "1 PLANTA", "floors": [1],
        "rooms": [
            {"name": "Estar Familiar",        "floor": 1, "label": 0, "area": 65},
            {"name": "Dormitorio Principal",  "floor": 1, "label": 2, "area": 32},
            {"name": "Baño Principal",        "floor": 1, "label": 7, "area":  8},
            {"name": "Vestidor A",            "floor": 1, "label": 3, "area": 10},
            {"name": "Vestidor B",            "floor": 1, "label": 3, "area":  8},
            {"name": "Dormitorio 2",          "floor": 1, "label": 1, "area": 24},
            {"name": "Baño 2",               "floor": 1, "label": 7, "area":  6},
            {"name": "Cocina",               "floor": 1, "label": 5, "area": 20},
            {"name": "Lavadero",             "floor": 1, "label": 5, "area":  8},
            {"name": "Baño Guest",           "floor": 1, "label": 7, "area":  5},
            {"name": "Terraza",              "floor": 1, "label": 0, "area": 60},
        ],
        "edges": [(0,1),(0,5),(0,7),(0,10),(1,2),(1,3),(1,4),(5,6),(7,8),(7,9)],
    },
    "104": {
        "unit_type": "DUPLEX", "floors": [0, 1],
        "rooms": [
            {"name": "Estar Familiar GF",      "floor": 0, "label": 0, "area": 55},
            {"name": "Pasillo GF",             "floor": 0, "label": 8, "area": 15},
            {"name": "Dormitorio Principal 1", "floor": 0, "label": 2, "area": 30},
            {"name": "Baño Principal 1",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 1",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Principal 2", "floor": 0, "label": 2, "area": 28},
            {"name": "Baño Principal 2",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 2",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Servicio",    "floor": 0, "label": 1, "area": 18},
            {"name": "Baño Servicio",          "floor": 0, "label": 7, "area":  6},
            {"name": "Lavadero GF",            "floor": 0, "label": 5, "area": 10},
            {"name": "Jardín Privado",         "floor": 0, "label": 0, "area": 80},
            {"name": "Estar F1",               "floor": 1, "label": 0, "area": 45},
            {"name": "Comedor",                "floor": 1, "label": 0, "area": 30},
            {"name": "Cocina F1",              "floor": 1, "label": 5, "area": 20},
            {"name": "Toilette F1",            "floor": 1, "label": 6, "area":  5},
            {"name": "Lavadero F1",            "floor": 1, "label": 5, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
        ],
        "edges": [
            (0,1),(0,11),(1,2),(1,5),(1,8),(1,10),
            (2,3),(2,4),(5,6),(5,7),(8,9),
            (12,13),(12,17),(13,14),(14,15),(14,16),
            (0,12),
        ],
    },
    "201": {
        "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 60},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 28},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  8},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 10},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 22},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  6},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 18},
            {"name": "Toilette F2",            "floor": 2, "label": 6, "area":  5},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 40},
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 50},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 15},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 10},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 75},
        ],
        "edges": [
            (0,1),(0,7),(0,9),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),
            (10,11),(10,12),(10,14),(12,13),
            (0,10),
        ],
    },
    "202": {
        "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 70},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 32},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  9},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 12},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 25},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  7},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 22},
            {"name": "Toilette F2",            "floor": 2, "label": 6, "area":  5},
            {"name": "Lavadero F2",            "floor": 2, "label": 5, "area":  9},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 45},
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 70},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 82},
        ],
        "edges": [
            (0,1),(0,7),(0,10),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),(7,9),
            (11,12),(11,13),(11,15),(13,14),
            (0,11),
        ],
    },
    "203": {
        "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 65},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 30},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  8},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 10},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 24},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  7},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 20},
            {"name": "Lavadero F2",            "floor": 2, "label": 5, "area":  8},
            {"name": "Baño Guest",             "floor": 2, "label": 7, "area":  5},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 45},
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 70},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 80},
        ],
        "edges": [
            (0,1),(0,7),(0,10),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),(7,9),
            (11,12),(11,13),(11,15),(13,14),
            (0,11),
        ],
    },
    "204": {
        "unit_type": "TRIPLEX", "floors": [1, 2, 3],
        "rooms": [
            {"name": "Dormitorio 1",           "floor": 1, "label": 1, "area": 30},
            {"name": "Baño 1 F1",              "floor": 1, "label": 7, "area":  8},
            {"name": "Dormitorio 2",           "floor": 1, "label": 1, "area": 28},
            {"name": "Baño 2 F1",              "floor": 1, "label": 7, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 80},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 35},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area": 10},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 22},
            {"name": "Baño Guest F2",          "floor": 2, "label": 7, "area":  6},
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 90},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 87},
        ],
        "edges": [
            (0,1),(0,4),(2,3),(2,4),
            (0,5),
            (5,6),(5,8),(5,9),(6,7),
            (5,10),
            (10,11),(10,12),(10,14),(12,13),
        ],
    },
}

print(f"Loaded {len(UNIT_ROOMS)} units — total rooms: {sum(len(u['rooms']) for u in UNIT_ROOMS.values())}")

In [ ]:
# Build one NetworkX graph per unit
def build_graph(unit_id, unit_data):
    G = nx.Graph(unit=unit_id, unit_type=unit_data["unit_type"])
    for nid, room in enumerate(unit_data["rooms"]):
        G.add_node(nid,
                   name=room["name"],
                   label=room["label"],
                   floor=room["floor"],
                   area=room["area"],
                   unit=unit_id)
    for (src, dst) in unit_data["edges"]:
        G.add_edge(src, dst)
    return G

GRAPHS = {uid: build_graph(uid, udata) for uid, udata in UNIT_ROOMS.items()}

print(f"{'Unit':<6} {'Type':<10} {'Nodes':>6} {'Edges':>6} {'Connected':>10}")
print("-" * 42)
for uid, G in GRAPHS.items():
    connected = nx.is_connected(G)
    print(f"{uid:<6} {UNIT_ROOMS[uid]['unit_type']:<10} {G.number_of_nodes():>6} {G.number_of_edges():>6} {str(connected):>10}")

## Section 2 — Degree Centrality

Degree centrality = `degree(node) / (n - 1)` where n is the total number of rooms in the unit.

**Architectural reading:** High degree = spatial hub. Corridors (Pasillo) and kitchens score highest because they connect to many rooms. Bathrooms and vestidores score lowest — they are accessed through a single room (leaf nodes).

In [ ]:
all_degree_rows = []
for uid, G in GRAPHS.items():
    dc = nx.degree_centrality(G)
    for nid, val in dc.items():
        d = G.nodes[nid]
        all_degree_rows.append({
            "unit": uid, "node_id": nid,
            "room": d["name"], "label": d["label"],
            "floor": d["floor"],
            "degree": G.degree(nid),
            "degree_centrality": round(val, 4),
        })

degree_df = pd.DataFrame(all_degree_rows)

print("Top 10 rooms by Degree Centrality across all units:")
top10 = degree_df.nlargest(10, "degree_centrality")[["unit", "room", "floor", "degree", "degree_centrality"]]
print(top10.to_string(index=False))

In [ ]:
# Subplot: degree centrality per unit
UNIT_LIST = list(UNIT_ROOMS.keys())
fig = make_subplots(rows=2, cols=4, subplot_titles=[f"Unit {u}" for u in UNIT_LIST],
                    shared_yaxes=True)

for idx, uid in enumerate(UNIT_LIST):
    r, c = divmod(idx, 4)
    sub = degree_df[degree_df["unit"] == uid].sort_values("degree_centrality", ascending=True)
    colors = [LABEL_COLORS[l] for l in sub["label"]]
    fig.add_trace(go.Bar(
        y=sub["room"].str[:20],
        x=sub["degree_centrality"],
        orientation="h",
        marker_color=colors,
        showlegend=False,
    ), row=r+1, col=c+1)

fig.update_layout(
    height=800, width=1400,
    title_text="Degree Centrality per Room — The Edge (all 8 units)",
    title_x=0.5,
)
fig.show(renderer=renderer)

## Section 3 — Closeness Centrality

Closeness centrality = `(n - 1) / sum_of_shortest_paths_to_all_others`

**Architectural reading:** High closeness = spatially central. A room you can reach every other room from in the fewest steps. In The Edge, the living room (Estar) is expected to score highest — it is the spatial heart that connects day, night, and service zones. Vestidores and terraces will score lowest (deep within the circulation tree).

In [ ]:
all_close_rows = []
for uid, G in GRAPHS.items():
    cc = nx.closeness_centrality(G)
    for nid, val in cc.items():
        d = G.nodes[nid]
        all_close_rows.append({
            "unit": uid, "node_id": nid,
            "room": d["name"], "label": d["label"],
            "floor": d["floor"],
            "closeness_centrality": round(val, 4),
        })

closeness_df = pd.DataFrame(all_close_rows)

print("Top 10 rooms by Closeness Centrality across all units:")
top10_cc = closeness_df.nlargest(10, "closeness_centrality")[["unit", "room", "floor", "closeness_centrality"]]
print(top10_cc.to_string(index=False))

In [ ]:
fig2 = make_subplots(rows=2, cols=4, subplot_titles=[f"Unit {u}" for u in UNIT_LIST],
                     shared_yaxes=True)

for idx, uid in enumerate(UNIT_LIST):
    r, c = divmod(idx, 4)
    sub = closeness_df[closeness_df["unit"] == uid].sort_values("closeness_centrality", ascending=True)
    colors = [LABEL_COLORS[l] for l in sub["label"]]
    fig2.add_trace(go.Bar(
        y=sub["room"].str[:20],
        x=sub["closeness_centrality"],
        orientation="h",
        marker_color=colors,
        showlegend=False,
    ), row=r+1, col=c+1)

fig2.update_layout(
    height=800, width=1400,
    title_text="Closeness Centrality per Room — The Edge (all 8 units)",
    title_x=0.5,
)
fig2.show(renderer=renderer)

## Section 4 — Clustering Coefficient

Clustering coefficient = fraction of a node's neighbours that are also neighbours of each other.

**Architectural reading:** A room with high clustering is embedded in a tightly connected sub-cluster — all its neighbours can also reach each other directly. In floor plans, service clusters (bathroom + vestidor + bedroom all connected) will score higher than corridor or living room nodes, which bridge between separate clusters.

In [ ]:
all_clust_rows = []
for uid, G in GRAPHS.items():
    clust = nx.clustering(G)
    for nid, val in clust.items():
        d = G.nodes[nid]
        all_clust_rows.append({
            "unit": uid, "node_id": nid,
            "room": d["name"], "label": d["label"],
            "floor": d["floor"],
            "clustering": round(val, 4),
        })

clust_df = pd.DataFrame(all_clust_rows)

# Average clustering per unit
unit_clust = clust_df.groupby("unit")["clustering"].mean().reset_index()
unit_clust.columns = ["Unit", "Avg Clustering"]
print("Average clustering coefficient per unit:")
print(unit_clust.to_string(index=False))
print()
print(f"Building average: {clust_df['clustering'].mean():.4f}")

In [ ]:
fig3 = make_subplots(rows=2, cols=4, subplot_titles=[f"Unit {u}" for u in UNIT_LIST],
                     shared_yaxes=True)

for idx, uid in enumerate(UNIT_LIST):
    r, c = divmod(idx, 4)
    sub = clust_df[clust_df["unit"] == uid].sort_values("clustering", ascending=True)
    colors = [LABEL_COLORS[l] for l in sub["label"]]
    fig3.add_trace(go.Bar(
        y=sub["room"].str[:20],
        x=sub["clustering"],
        orientation="h",
        marker_color=colors,
        showlegend=False,
    ), row=r+1, col=c+1)

fig3.update_layout(
    height=800, width=1400,
    title_text="Clustering Coefficient per Room — The Edge (all 8 units)",
    title_x=0.5,
)
fig3.show(renderer=renderer)

## Section 5 — Shortest Paths

Shortest path length = minimum number of rooms you must pass through to travel between any two rooms.

Two analyses:
1. **Average shortest path length per unit** — overall compactness of the unit plan
2. **Depth map from the living room (Estar)** — how many steps from the main living area to every other room (spatial depth sequence)

In [ ]:
# 1. Average shortest path length per unit (only for connected graphs)
print("Average Shortest Path Length per Unit:")
print("-" * 40)
sp_summary = []
for uid, G in GRAPHS.items():
    if nx.is_connected(G):
        avg_sp = nx.average_shortest_path_length(G)
        diameter = nx.diameter(G)
        sp_summary.append({"Unit": uid, "Type": UNIT_ROOMS[uid]["unit_type"],
                            "Rooms": G.number_of_nodes(),
                            "Avg Path Length": round(avg_sp, 3),
                            "Diameter (max path)": diameter})
    else:
        # Handle disconnected graph — compute per component
        comps = list(nx.connected_components(G))
        avg_sp = np.mean([nx.average_shortest_path_length(G.subgraph(c)) for c in comps])
        sp_summary.append({"Unit": uid, "Type": UNIT_ROOMS[uid]["unit_type"],
                            "Rooms": G.number_of_nodes(),
                            "Avg Path Length": round(avg_sp, 3),
                            "Diameter (max path)": "N/A (disconnected)"})

sp_df = pd.DataFrame(sp_summary)
print(sp_df.to_string(index=False))

In [ ]:
# Average shortest path bar chart
sp_plot = sp_df[sp_df["Avg Path Length"].notna()].copy()
sp_plot["Avg Path Length"] = pd.to_numeric(sp_plot["Avg Path Length"])

fig4 = px.bar(
    sp_plot, x="Unit", y="Avg Path Length",
    color="Avg Path Length", color_continuous_scale="Viridis_r",
    text=sp_plot["Avg Path Length"].map("{:.2f}".format),
    labels={"Avg Path Length": "Avg. Shortest Path (rooms)"},
    title="Average Shortest Path Length per Unit — The Edge<br><sup>Lower = more compact plan; higher = deeper spatial sequence</sup>",
)
fig4.update_traces(textposition="outside")
fig4.update_layout(width=800, height=450, coloraxis_showscale=False)
fig4.show(renderer=renderer)

In [ ]:
# 2. Depth map from the living room (Estar) — one subplot per unit
# Find node 0 (always the main Estar / first bedroom for unit 204)

fig5 = make_subplots(rows=2, cols=4, subplot_titles=[f"Unit {u} (depth from room 0)" for u in UNIT_LIST],
                     shared_yaxes=True)

for idx, uid in enumerate(UNIT_LIST):
    r, c = divmod(idx, 4)
    G = GRAPHS[uid]
    # shortest paths from node 0 (Estar / first room)
    try:
        sp_from_0 = nx.single_source_shortest_path_length(G, 0)
    except Exception:
        continue

    rooms = UNIT_ROOMS[uid]["rooms"]
    sub_rows = []
    for nid, depth in sp_from_0.items():
        sub_rows.append({
            "room": rooms[nid]["name"],
            "depth": depth,
            "label": rooms[nid]["label"],
        })
    sub = pd.DataFrame(sub_rows).sort_values("depth")

    colors = [LABEL_COLORS[l] for l in sub["label"]]
    fig5.add_trace(go.Bar(
        y=sub["room"].str[:20],
        x=sub["depth"],
        orientation="h",
        marker_color=colors,
        showlegend=False,
    ), row=r+1, col=c+1)

fig5.update_layout(
    height=800, width=1400,
    title_text="Spatial Depth from Main Room (steps) — The Edge<br><sup>How many rooms you must pass through to reach each space</sup>",
    title_x=0.5,
)
fig5.show(renderer=renderer)

## Section 6 — Network Graph Visualisation

Node-link diagram per unit. Node size = area (m²), node colour = room type zone, edge = door connection.

In [ ]:
def plot_unit_graph(uid, G, metric_key=None, metric_dict=None, title_suffix=""):
    """
    Draw a unit's room graph with Plotly.
    Nodes positioned by floor (y) and node_id (x).
    metric_dict: optional {node_id: float} to size/colour nodes by metric value.
    """
    rooms = UNIT_ROOMS[uid]["rooms"]
    floors = sorted(set(r["floor"] for r in rooms))
    floor_map = {f: i for i, f in enumerate(floors)}

    # Assign x positions per floor
    floor_counters = {f: 0 for f in floors}
    pos = {}
    for nid in G.nodes():
        fl = rooms[nid]["floor"]
        pos[nid] = (floor_counters[fl] * 1.8, floor_map[fl] * 2.0)
        floor_counters[fl] += 1

    # Edges
    edge_x, edge_y = [], []
    for src, dst in G.edges():
        x0, y0 = pos[src]
        x1, y1 = pos[dst]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    # Nodes
    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]
    node_colors = [LABEL_COLORS[rooms[n]["label"]] for n in G.nodes()]
    node_sizes  = [max(12, rooms[n]["area"] * 0.5) for n in G.nodes()]
    node_text   = [f"{rooms[n]['name']}<br>Floor {rooms[n]['floor']}<br>Area {rooms[n]['area']}m²" for n in G.nodes()]
    node_labels = [rooms[n]["name"][:18] for n in G.nodes()]

    if metric_dict:
        vals = [metric_dict.get(n, 0) for n in G.nodes()]
        node_colors = vals  # will use colorscale
        node_text   = [f"{node_text[i]}<br>{metric_key}: {vals[i]:.3f}" for i in range(len(node_text))]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(color="#cccccc", width=2), hoverinfo="none",
    ))
    node_trace_kwargs = dict(
        x=node_x, y=node_y, mode="markers+text",
        text=node_labels, textposition="top center",
        marker=dict(size=node_sizes, line=dict(color="white", width=1)),
        hovertext=node_text, hoverinfo="text",
    )
    if metric_dict:
        node_trace_kwargs["marker"]["color"] = vals
        node_trace_kwargs["marker"]["colorscale"] = "RdYlGn"
        node_trace_kwargs["marker"]["showscale"] = True
        node_trace_kwargs["marker"]["colorbar"] = dict(title=metric_key)
    else:
        node_trace_kwargs["marker"]["color"] = node_colors

    fig.add_trace(go.Scatter(**node_trace_kwargs))

    floor_names = ["GF", "F1", "F2", "Roof"]
    for fl, fi in floor_map.items():
        fig.add_annotation(x=-0.5, y=fi * 2.0,
                           text=f"<b>{floor_names[fl]}</b>",
                           showarrow=False, font=dict(size=11, color="grey"))

    fig.update_layout(
        title=f"Unit {uid} — {UNIT_ROOMS[uid]['unit_type']} {title_suffix}",
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=700, height=420,
        plot_bgcolor="#fafafa",
    )
    return fig


# Show all 8 unit graphs coloured by room zone type
for uid, G in GRAPHS.items():
    fig = plot_unit_graph(uid, G, title_suffix="— Room Zones")
    fig.show(renderer=renderer)

In [ ]:
# Re-draw coloured by Closeness Centrality
for uid, G in GRAPHS.items():
    cc_dict = nx.closeness_centrality(G)
    fig = plot_unit_graph(uid, G, metric_key="Closeness",
                          metric_dict=cc_dict,
                          title_suffix="— Closeness Centrality")
    fig.show(renderer=renderer)

## Section 7 — Building-Level Summary

Compare all 8 units on the four graph metrics to understand structural differences between unit types.

In [ ]:
summary_rows = []
for uid, G in GRAPHS.items():
    dc = nx.degree_centrality(G)
    cc = nx.closeness_centrality(G)
    cl = nx.clustering(G)

    # Find most central room by each metric
    rooms = UNIT_ROOMS[uid]["rooms"]
    max_dc_node = max(dc, key=dc.get)
    max_cc_node = max(cc, key=cc.get)
    max_cl_node = max(cl, key=cl.get)

    if nx.is_connected(G):
        avg_sp   = round(nx.average_shortest_path_length(G), 3)
        diameter = nx.diameter(G)
    else:
        comps = list(nx.connected_components(G))
        avg_sp   = round(np.mean([nx.average_shortest_path_length(G.subgraph(c)) for c in comps]), 3)
        diameter = "—"

    summary_rows.append({
        "Unit":          uid,
        "Type":          UNIT_ROOMS[uid]["unit_type"],
        "Rooms":         G.number_of_nodes(),
        "Edges":         G.number_of_edges(),
        "Max Degree Hub":     rooms[max_dc_node]["name"],
        "Max Degree Val":     round(dc[max_dc_node], 3),
        "Most Central Room":  rooms[max_cc_node]["name"],
        "Closeness Val":      round(cc[max_cc_node], 3),
        "Max Clustering Room":rooms[max_cl_node]["name"],
        "Avg Path Length":    avg_sp,
        "Diameter":           diameter,
    })

summary_df = pd.DataFrame(summary_rows)

# Display in two parts for readability
print("=== STRUCTURAL METRICS ===")
print(summary_df[["Unit","Type","Rooms","Edges","Avg Path Length","Diameter"]].to_string(index=False))
print()
print("=== TOP ROOMS PER METRIC ===")
print(summary_df[["Unit","Max Degree Hub","Max Degree Val","Most Central Room","Closeness Val"]].to_string(index=False))

In [ ]:
# Comparison: avg path length + avg closeness per unit
comp_rows = []
for uid, G in GRAPHS.items():
    cc_vals = list(nx.closeness_centrality(G).values())
    dc_vals = list(nx.degree_centrality(G).values())
    cl_vals = list(nx.clustering(G).values())
    comp_rows.append({
        "Unit":           uid,
        "Type":           UNIT_ROOMS[uid]["unit_type"],
        "Avg Closeness":  round(np.mean(cc_vals), 4),
        "Avg Degree":     round(np.mean(dc_vals), 4),
        "Avg Clustering": round(np.mean(cl_vals), 4),
    })
comp_df = pd.DataFrame(comp_rows)

fig6 = make_subplots(rows=1, cols=3, subplot_titles=[
    "Avg Closeness Centrality", "Avg Degree Centrality", "Avg Clustering Coefficient"
])

for col_idx, (col, title) in enumerate([
    ("Avg Closeness", "Closeness"),
    ("Avg Degree",    "Degree"),
    ("Avg Clustering","Clustering"),
]):
    fig6.add_trace(go.Bar(
        x=comp_df["Unit"],
        y=comp_df[col],
        text=comp_df[col].map("{:.3f}".format),
        textposition="outside",
        marker_color=px.colors.qualitative.Set2[:8],
        showlegend=False,
    ), row=1, col=col_idx+1)

fig6.update_layout(
    height=420, width=1100,
    title_text="Average Graph Metrics per Unit — The Edge",
    title_x=0.5,
)
fig6.show(renderer=renderer)

## Section 8 — Architectural Findings

### Degree Centrality
- **Pasillo (corridor)** nodes in units 101, 104, 201–203 score highest — they are the mandatory connectors between the sleeping zone and the living zone
- **Estar Familiar (living room)** scores highest in single-floor units (102, 103) because there is no corridor and the living room directly connects all program elements
- **Bathrooms, vestidores, terraces** score lowest — they are dead-end leaf nodes accessed only through one room

### Closeness Centrality
- Rooms on the **intermediate level** of multi-storey units score highest: in a duplex, the floor that bridges both levels (via the private stair) is the most spatially central
- Unit 204 (triplex): F2 is the spatial spine — it is 2 steps from both the F1 bedrooms and the rooftop, and directly contains the main living room
- Terraces and rooftop elements score lowest — they sit at the periphery of the circulation tree

### Clustering Coefficient
- **Bedroom clusters** (bedroom + ensuite + vestidor all connected) produce a high local clustering coefficient for the bedroom node
- **Living rooms and corridors** score near 0 — they connect to rooms that are NOT connected to each other (bedroom ≠ connected to kitchen directly), making them bridges rather than cluster members
- This pattern confirms the classic residential plan tripartition: day zone cluster / night zone cluster / service cluster, bridged by circulation

### Shortest Paths
- **1 PLANTA units (102, 103)** have shorter average paths — all rooms on one floor, compact layout
- **TRIPLEX unit 204** has the longest average path — rooms span 3 floors requiring mandatory stair traversal
- **Spatial depth from Estar:** vestidores and second bathrooms consistently require 3–4 steps (deep private spaces), while kitchen requires 1–2 steps (close to day zone)